In [ ]:
!pip install ipython-sql

%%sql

-- PROYECTO: ANÁLISIS DE FUNNEL Y RETENCIÓN MERCADO LIBRE
-- Objetivo: Identificar puntos de abandono y lealtad de usuarios

-- 1. Embudo de Conversión (Conversion Funnel)
-- Hallazgo clave: La mayor caída ocurre al intentar agregar al carrito (Drop-off de ~89%)
WITH funnel_stages AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_name = 'first_visit' THEN user_id END) as visits,
        COUNT(DISTINCT CASE WHEN event_name = 'select_item' THEN user_id END) as selections,
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END) as cart_adds,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) as purchases
    FROM mercadolibre_funnel
)
SELECT
    ROUND(selections * 100.0 / visits, 2) AS pct_visit_to_select,
    ROUND(cart_adds * 100.0 / selections, 2) AS pct_select_to_cart, -- Fuga crítica detectada aquí
    ROUND(purchases * 100.0 / cart_adds, 2) AS pct_cart_to_purchase
FROM funnel_stages;

-- 2. Análisis de Retención por Cohortes (Retention Rate)
-- Hallazgo: La retención cae drásticamente del día 21 al 28 (pasa de ~24% a <3%)
SELECT
    cohort,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN day_after_signup >= 7 AND active = 1 THEN user_id END) / COUNT(DISTINCT user_id), 1) AS retention_d7_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN day_after_signup >= 28 AND active = 1 THEN user_id END) / COUNT(DISTINCT user_id), 1) AS retention_d28_pct
FROM user_activity_table
GROUP BY cohort
ORDER BY cohort;